In [0]:
# ============================================================
# Setup: Databricks SDK workspace client
# ============================================================
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineState
from datetime import datetime
import json
import time

w = WorkspaceClient()

# Constants
JOB_ID = 19110422057552
PIPELINE_ID = None  # This job uses notebooks, not a pipeline
CATALOG = "nyc_mobility"
SCHEMA = "gold"

print(f"Workspace client initialized. JOB_ID={JOB_ID}, PIPELINE_ID={PIPELINE_ID}")

Workspace client initialized. JOB_ID=19110422057552, PIPELINE_ID=None


In [0]:
# ============================================================
# Section 1: Fetch Job Run History via Jobs API
# ============================================================
runs = w.jobs.list_runs(job_id=JOB_ID)

job_runs_data = []
for run in runs:
    job_runs_data.append({
        "run_id": str(run.run_id) if run.run_id else None,
        "job_id": str(JOB_ID),
        "run_name": run.run_name or None,
        "state": run.state.life_cycle_state.value if run.state and run.state.life_cycle_state else None,
        "result_state": run.state.result_state.value if run.state and run.state.result_state else None,
        "start_time": datetime.fromtimestamp(run.start_time / 1000) if run.start_time else None,
        "end_time": datetime.fromtimestamp(run.end_time / 1000) if run.end_time else None,
        "duration_ms": run.run_duration if hasattr(run, 'run_duration') and run.run_duration else None,
        "trigger": str(run.trigger) if hasattr(run, 'trigger') and run.trigger else None,
        "creator": run.creator_user_name or None,
        "run_page_url": run.run_page_url or None,
        "number_in_job": run.number_in_job if hasattr(run, 'number_in_job') else None,
    })

print(f"Fetched {len(job_runs_data)} job runs")

# Write to Delta table
spark.createDataFrame(job_runs_data).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.job_run_history")

print(f"Saved {len(job_runs_data)} rows to {CATALOG}.{SCHEMA}.job_run_history")

Fetched 5 job runs
Saved 5 rows to nyc_mobility.gold.job_run_history


In [0]:
# ============================================================
# Section 2: Fetch Task Run History via Jobs API (with expanded tasks)
# ============================================================
task_runs_data = []
try:
    runs_expanded = list(w.jobs.list_runs(job_id=JOB_ID, expand_tasks=True))
except TypeError:
    # Fallback if expand_tasks not supported
    runs_expanded = list(w.jobs.list_runs(job_id=JOB_ID))

for run in runs_expanded:
    if hasattr(run, 'tasks') and run.tasks:
        for task in run.tasks:
            task_runs_data.append({
                "run_id": str(run.run_id) if run.run_id else None,
                "job_id": str(JOB_ID),
                "task_key": task.task_key if hasattr(task, 'task_key') else None,
                "task_run_id": str(task.run_id) if hasattr(task, 'run_id') and task.run_id else None,
                "state": task.state.life_cycle_state.value if hasattr(task, 'state') and task.state and hasattr(task.state, 'life_cycle_state') and task.state.life_cycle_state else None,
                "result_state": task.state.result_state.value if hasattr(task, 'state') and task.state and hasattr(task.state, 'result_state') and task.state.result_state else None,
                "start_time": datetime.fromtimestamp(task.start_time / 1000) if hasattr(task, 'start_time') and task.start_time else None,
                "end_time": datetime.fromtimestamp(task.end_time / 1000) if hasattr(task, 'end_time') and task.end_time else None,
                "duration_ms": task.task_duration if hasattr(task, 'task_duration') and task.task_duration else None,
            })

print(f"Fetched {len(task_runs_data)} task runs")

if task_runs_data:
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    
    task_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("job_id", StringType(), True),
        StructField("task_key", StringType(), True),
        StructField("task_run_id", StringType(), True),
        StructField("state", StringType(), True),
        StructField("result_state", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_ms", LongType(), True),
    ])
    
    spark.createDataFrame(task_runs_data, schema=task_schema).write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.task_run_history")
    print(f"Saved {len(task_runs_data)} rows to {CATALOG}.{SCHEMA}.task_run_history")
else:
    print("No task runs found")

Fetched 57 task runs
Saved 57 rows to nyc_mobility.gold.task_run_history


In [0]:
# ============================================================
# Section 3: Fetch Pipeline Update History via Pipelines API
# Only runs if PIPELINE_ID is set (this job uses notebooks, not a pipeline)
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

pipeline_data = []

if PIPELINE_ID:
    response = w.pipelines.list_updates(pipeline_id=PIPELINE_ID)
    update_list = response.updates if hasattr(response, 'updates') and response.updates else []

    explicit_schema = StructType([
        StructField("update_id", StringType(), True),
        StructField("pipeline_id", StringType(), True),
        StructField("state", StringType(), True),
        StructField("update_type", StringType(), True),
        StructField("creation_time", StringType(), True),
        StructField("started_at", StringType(), True),
        StructField("completed_at", StringType(), True),
        StructField("duration_seconds", IntegerType(), True),
        StructField("config_json", StringType(), True),
        StructField("cause", StringType(), True),
    ])

    # Build start/end timestamps from pipeline events
    update_times = {}
    events = w.pipelines.list_pipeline_events(pipeline_id=PIPELINE_ID)
    for ev in events:
        if ev.event_type == 'update_progress' and ev.origin and ev.origin.update_id:
            uid = ev.origin.update_id
            if uid not in update_times:
                update_times[uid] = {"started": None, "completed": None}
            msg = (ev.message or "").lower()
            ts = ev.timestamp
            if 'start' in msg or 'running' in msg or 'deploying' in msg:
                if not update_times[uid]["started"]:
                    update_times[uid]["started"] = ts
            if 'completed' in msg or 'failed' in msg:
                update_times[uid]["completed"] = ts
    print(f"Extracted timestamps for {len(update_times)} updates from pipeline events")

    for update in update_list:
        uid = str(update.update_id) if update.update_id else None
        times = update_times.get(uid, {}) if uid else {}
        started = times.get("started")
        completed = times.get("completed")
        duration_seconds = None
        
        if started and completed:
            try:
                dt_start = datetime.fromisoformat(started.replace('Z', '+00:00'))
                dt_complete = datetime.fromisoformat(completed.replace('Z', '+00:00'))
                duration_seconds = int((dt_complete - dt_start).total_seconds())
            except Exception as e:
                print(f"Duration calc error for update {uid}: {e}")
        
        ct_iso = None
        if hasattr(update, 'creation_time') and update.creation_time:
            try:
                ct_iso = datetime.fromtimestamp(update.creation_time / 1000).isoformat()
            except Exception:
                ct_iso = str(update.creation_time)
        
        utype = "FULL_REFRESH" if getattr(update, 'full_refresh', False) else "INCREMENTAL"
        
        pipeline_data.append({
            "update_id": uid,
            "pipeline_id": PIPELINE_ID,
            "state": update.state.value if hasattr(update, 'state') and update.state else None,
            "update_type": utype,
            "creation_time": ct_iso,
            "started_at": started,
            "completed_at": completed,
            "duration_seconds": duration_seconds,
            "config_json": str(update.config) if hasattr(update, 'config') and update.config else None,
            "cause": update.cause.value if hasattr(update, 'cause') and update.cause else None,
        })

    print(f"Fetched {len(pipeline_data)} pipeline updates")

    if pipeline_data:
        spark.createDataFrame(pipeline_data, schema=explicit_schema).write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{CATALOG}.{SCHEMA}.pipeline_update_history")
        print(f"Saved {len(pipeline_data)} rows to {CATALOG}.{SCHEMA}.pipeline_update_history")
    else:
        print("No pipeline updates found")
else:
    print("PIPELINE_ID is None — this job uses notebooks, not a pipeline. Skipping pipeline update history.")

PIPELINE_ID is None — this job uses notebooks, not a pipeline. Skipping pipeline update history.


In [0]:
# ============================================================
# Summary: Display what was fetched
# ============================================================
print("=" * 60)
print("MONITORING DATA FETCH COMPLETE")
print("=" * 60)
print(f"Job runs: {len(job_runs_data)} → {CATALOG}.{SCHEMA}.job_run_history")
print(f"Task runs: {len(task_runs_data)} → {CATALOG}.{SCHEMA}.task_run_history")
print(f"Pipeline updates: {len(pipeline_data)} → {CATALOG}.{SCHEMA}.pipeline_update_history")
print()

# Display sample of pipeline update history
if PIPELINE_ID and pipeline_data:
    print("\nPipeline Update History (first 5):")
    display(spark.sql(f"SELECT update_id, state, update_type, started_at, completed_at, duration_seconds FROM {CATALOG}.{SCHEMA}.pipeline_update_history LIMIT 5"))

MONITORING DATA FETCH COMPLETE
Job runs: 5 → nyc_mobility.gold.job_run_history
Task runs: 57 → nyc_mobility.gold.task_run_history
Pipeline updates: 0 → nyc_mobility.gold.pipeline_update_history



In [0]:
# ============================================================
# 8.1 EXECUTION MONITORING
# Did the job run? When? How long? Did each step complete?
# ============================================================
print("=" * 60)
print("8.1 EXECUTION MONITORING")
print("=" * 60)

# --- Job Run Summary ---
print("\n--- Job Run History ---")
display(spark.sql(f"""
    SELECT 
        run_name,
        state AS life_cycle_state,
        result_state,
        start_time,
        end_time,
        CASE WHEN duration_ms IS NOT NULL 
            THEN CONCAT(CAST(duration_ms / 60000 AS DECIMAL(10,1)), ' min')
            ELSE NULL END AS duration,
        trigger,
        creator
    FROM {CATALOG}.{SCHEMA}.job_run_history
    ORDER BY start_time DESC
"""))

# --- Pipeline Update Summary ---
if PIPELINE_ID:
    print("\n--- Pipeline Update History (last 10) ---")
    display(spark.sql(f"""
        SELECT 
            update_id,
            state,
            update_type,
            started_at,
            completed_at,
            duration_seconds,
            cause
        FROM {CATALOG}.{SCHEMA}.pipeline_update_history
        ORDER BY started_at DESC
        LIMIT 10
    """))
else:
    print("\n--- Pipeline Update History: Skipped (notebook-based job, no pipeline) ---")

# --- Task Run Summary ---
print("\n--- Task Run History ---")
try:
    task_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.{SCHEMA}.task_run_history").collect()[0]['cnt']
    if task_count > 0:
        display(spark.sql(f"""
            SELECT 
                task_key,
                state,
                result_state,
                start_time,
                end_time,
                duration_ms
            FROM {CATALOG}.{SCHEMA}.task_run_history
            ORDER BY start_time DESC
        """))
    else:
        print("No task runs found in task_run_history table")
except Exception as e:
    print(f"task_run_history table not available: {e}")

# --- Execution Summary Stats ---
print("\n--- Execution Summary ---")
if PIPELINE_ID:
    display(spark.sql(f"""
        SELECT 
            COUNT(*) as total_pipeline_updates,
            SUM(CASE WHEN state = 'COMPLETED' THEN 1 ELSE 0 END) as completed,
            SUM(CASE WHEN state = 'FAILED' THEN 1 ELSE 0 END) as failed,
            ROUND(AVG(duration_seconds), 1) as avg_duration_seconds,
            MAX(started_at) as latest_run
        FROM {CATALOG}.{SCHEMA}.pipeline_update_history
    """))
else:
    display(spark.sql(f"""
        SELECT 
            COUNT(*) as total_job_runs,
            SUM(CASE WHEN result_state = 'SUCCESS' THEN 1 ELSE 0 END) as successful,
            SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) as failed,
            MAX(start_time) as latest_run
        FROM {CATALOG}.{SCHEMA}.job_run_history
    """))

In [0]:
# ============================================================
# 8.2 FAILURES MONITORING
# Which job/step failed? When? What was the error? How often?
# ============================================================
print("=" * 60)
print("8.2 FAILURES MONITORING")
print("=" * 60)

# --- Failed Job Runs ---
print("\n--- Failed Job Runs ---")
try:
    failed_jobs = spark.sql(f"""
        SELECT 
            run_id,
            run_name,
            result_state,
            start_time,
            end_time,
            creator
        FROM {CATALOG}.{SCHEMA}.job_run_history
        WHERE result_state = 'FAILED'
        ORDER BY start_time DESC
    """)
    if failed_jobs.count() > 0:
        display(failed_jobs)
    else:
        print("No failed job runs found")
except Exception as e:
    print(f"Error querying job failures: {e}")

# --- Failed Pipeline Updates ---
if PIPELINE_ID:
    print("\n--- Failed Pipeline Updates ---")
    try:
        failed_pipelines = spark.sql(f"""
            SELECT 
                update_id,
                state,
                update_type,
                started_at,
                completed_at,
                cause
            FROM {CATALOG}.{SCHEMA}.pipeline_update_history
            WHERE state = 'FAILED'
            ORDER BY started_at DESC
        """)
        if failed_pipelines.count() > 0:
            display(failed_pipelines)
        else:
            print("No failed pipeline updates found")
    except Exception as e:
        print(f"Error querying pipeline failures: {e}")
else:
    print("\n--- Failed Pipeline Updates: Skipped (notebook-based job) ---")

# --- Failed Task Runs (notebook-based) ---
print("\n--- Failed Task Runs ---")
try:
    failed_tasks = spark.sql(f"""
        SELECT 
            run_id,
            task_key,
            result_state,
            start_time,
            end_time
        FROM {CATALOG}.{SCHEMA}.task_run_history
        WHERE result_state = 'FAILED'
        ORDER BY start_time DESC
    """)
    if failed_tasks.count() > 0:
        display(failed_tasks)
    else:
        print("No failed task runs found")
except Exception as e:
    print(f"Error querying task failures: {e}")

# --- Failure Rate Summary ---
print("\n--- Failure Rate Summary ---")
if PIPELINE_ID:
    display(spark.sql(f"""
        SELECT 
            COUNT(*) as total_updates,
            SUM(CASE WHEN state = 'COMPLETED' THEN 1 ELSE 0 END) as completed,
            SUM(CASE WHEN state = 'FAILED' THEN 1 ELSE 0 END) as failed,
            ROUND(
                SUM(CASE WHEN state = 'FAILED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
                1
            ) as failure_rate_pct
        FROM {CATALOG}.{SCHEMA}.pipeline_update_history
    """))
else:
    display(spark.sql(f"""
        SELECT 
            COUNT(*) as total_runs,
            SUM(CASE WHEN result_state = 'SUCCESS' THEN 1 ELSE 0 END) as successful,
            SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) as failed,
            ROUND(
                SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
                1
            ) as failure_rate_pct
        FROM {CATALOG}.{SCHEMA}.job_run_history
    """))

In [0]:
# ============================================================
# 8.3 FRESHNESS MONITORING
# Is the data up to date? Check latest data in gold tables.
# ============================================================
print("=" * 60)
print("8.3 FRESHNESS MONITORING")
print("=" * 60)

from datetime import datetime
from pyspark.sql.functions import max as spark_max, current_date, datediff, col

check_time = datetime.now()
print(f"Freshness check run at: {check_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Discover gold tables (exclude monitoring tables)
monitoring_tables = ['job_run_history', 'task_run_history', 'pipeline_update_history']

try:
    gold_tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
except Exception:
    gold_tables = []

freshness_results = []
for row in gold_tables:
    table_name = row.tableName
    if table_name in monitoring_tables:
        continue
    
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_table}").collect()
        ts_col = None
        date_col = None
        for c in cols:
            col_name = c.col_name
            col_type = c.data_type
            if col_name.startswith('#'):
                continue
            if 'timestamp' in str(col_type).lower() and not ts_col:
                ts_col = col_name
            if col_type == 'date' and not date_col:
                date_col = col_name
        
        latest_value = None
        freshness_col = None
        if ts_col:
            latest_value = spark.sql(f"SELECT MAX({ts_col}) as latest FROM {full_table}").collect()[0]['latest']
            freshness_col = ts_col
        elif date_col:
            latest_value = spark.sql(f"SELECT MAX({date_col}) as latest FROM {full_table}").collect()[0]['latest']
            freshness_col = date_col
        
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table}").collect()[0]['cnt']
        
        # Calculate staleness in days
        days_stale = None
        if latest_value:
            try:
                if hasattr(latest_value, 'date'):
                    days_stale = (datetime.now().date() - latest_value.date()).days
                else:
                    days_stale = (datetime.now() - latest_value).days
            except Exception:
                pass
        
        status = "FRESH" if days_stale is not None and days_stale <= 1 else ("STALE" if days_stale is not None and days_stale > 1 else "UNKNOWN")
        
        freshness_results.append({
            "table_name": table_name,
            "freshness_column": freshness_col or "N/A",
            "latest_data": str(latest_value) if latest_value else "N/A",
            "days_stale": days_stale,
            "status": status,
            "row_count": row_count,
            "checked_at": check_time,
        })
    except Exception as e:
        freshness_results.append({
            "table_name": table_name,
            "freshness_column": "ERROR",
            "latest_data": str(e)[:80],
            "days_stale": None,
            "status": "ERROR",
            "row_count": 0,
            "checked_at": check_time,
        })

if freshness_results:
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
    freshness_schema = StructType([
        StructField("table_name", StringType(), True),
        StructField("freshness_column", StringType(), True),
        StructField("latest_data", StringType(), True),
        StructField("days_stale", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("row_count", LongType(), True),
        StructField("checked_at", StringType(), True),
    ])
    display(spark.createDataFrame(freshness_results, schema=freshness_schema))
else:
    print("No data tables found in gold schema for freshness check")

8.3 FRESHNESS MONITORING
Freshness check run at: 2026-09-25 11:30:19


table_name,freshness_column,latest_data,days_stale,status,row_count,checked_at
dim_date,full_date,2026-06-01,null,UNKNOWN,93,2026-09-25 11:30:19.272696
dim_hour,N/A,N/A,null,UNKNOWN,24,2026-09-25 11:30:19.272696
dim_weather,N/A,N/A,null,UNKNOWN,19,2026-09-25 11:30:19.272696
dim_zone,N/A,N/A,null,UNKNOWN,265,2026-09-25 11:30:19.272696
dq_check_results,checked_at,2026-09-25 10:51:51.945636,0,FRESH,15,2026-09-25 11:30:19.272696
fact_trip,N/A,N/A,null,UNKNOWN,133344,2026-09-25 11:30:19.272696


In [0]:
# ============================================================
# 8.4 DATA QUALITY MONITORING
# Missing values, duplicates, invalid values, record counts
# ============================================================
print("=" * 60)
print("8.4 DATA QUALITY MONITORING")
print("=" * 60)

# Reuse gold table list (exclude monitoring tables)
monitoring_tables = ['job_run_history', 'task_run_history', 'pipeline_update_history']

try:
    gold_tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
except Exception:
    gold_tables = []

quality_results = []
for row in gold_tables:
    table_name = row.tableName
    if table_name in monitoring_tables:
        continue
    
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_table}").collect()
        col_names = [c.col_name for c in cols if not c.col_name.startswith('#')]
        
        # Total rows
        total_rows = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_table}").collect()[0]['cnt']
        
        # Count nulls in first 10 columns
        nulls_found = []
        for cn in col_names[:10]:
            null_cnt = spark.sql(f"SELECT SUM(CASE WHEN `{cn}` IS NULL THEN 1 ELSE 0 END) as nulls FROM {full_table}").collect()[0]['nulls']
            if null_cnt and null_cnt > 0:
                nulls_found.append(f"{cn}={null_cnt}")
        
        # Duplicate check (first 10 columns)
        check_cols = [f"`{c}`" for c in col_names[:10]]
        dup_count = spark.sql(f"""
            SELECT COUNT(*) as dups FROM (
                SELECT {', '.join(check_cols)}, COUNT(*) as cnt
                FROM {full_table}
                GROUP BY {', '.join(check_cols)}
                HAVING COUNT(*) > 1
            )
        """).collect()[0]['dups']
        
        quality_results.append({
            "table_name": table_name,
            "total_rows": total_rows,
            "columns_checked": len(col_names),
            "duplicate_groups": dup_count,
            "nulls_summary": ", ".join(nulls_found) if nulls_found else "No nulls",
        })
    except Exception as e:
        quality_results.append({
            "table_name": table_name,
            "total_rows": 0,
            "columns_checked": 0,
            "duplicate_groups": 0,
            "nulls_summary": f"ERROR: {str(e)[:60]}",
        })

if quality_results:
    display(spark.createDataFrame(quality_results))
else:
    print("No data tables found in gold schema for quality check")

8.4 DATA QUALITY MONITORING


columns_checked,duplicate_groups,nulls_summary,table_name,total_rows
11,0,No nulls,dim_date,93
4,0,No nulls,dim_hour,24
5,0,No nulls,dim_weather,19
5,0,No nulls,dim_zone,265
17,0,passenger_count=18753,fact_trip,133344


In [0]:
# ============================================================
# 8.5 SAVE DATA QUALITY RESULTS TO TABLE
# Transform quality_results into check-based format and save
# ============================================================
print("=" * 60)
print("8.5 SAVING DATA QUALITY RESULTS")
print("=" * 60)

from datetime import datetime

# Transform quality_results into individual check records
dq_check_records = []
current_timestamp = datetime.now()

for result in quality_results:
    table_name = result['table_name']
    total_rows = result['total_rows']
    duplicate_groups = result['duplicate_groups']
    nulls_summary = result['nulls_summary']
    
    # Check 1: Row Count
    dq_check_records.append({
        "check_id": f"{table_name}_row_count",
        "check_type": "completeness",
        "table_name": table_name,
        "check_name": "Row Count",
        "check_description": f"Total rows in {table_name}",
        "status": "PASS" if total_rows > 0 else "FAIL",
        "failed_count": 0 if total_rows > 0 else 1,
        "total_count": 1,
        "pass_rate": 1.0 if total_rows > 0 else 0.0,
        "severity": "critical",
        "checked_at": current_timestamp,
    })
    
    # Check 2: Duplicate Check
    dq_check_records.append({
        "check_id": f"{table_name}_duplicates",
        "check_type": "uniqueness",
        "table_name": table_name,
        "check_name": "Duplicate Check",
        "check_description": f"Duplicate groups in {table_name}",
        "status": "PASS" if duplicate_groups == 0 else "WARN",
        "failed_count": duplicate_groups,
        "total_count": total_rows,
        "pass_rate": 1.0 if duplicate_groups == 0 else (1.0 - (duplicate_groups / max(total_rows, 1))),
        "severity": "medium",
        "checked_at": current_timestamp,
    })
    
    # Check 3: Null Check
    has_nulls = nulls_summary != "No nulls" and not nulls_summary.startswith("ERROR")
    dq_check_records.append({
        "check_id": f"{table_name}_nulls",
        "check_type": "completeness",
        "table_name": table_name,
        "check_name": "Null Check",
        "check_description": nulls_summary if has_nulls else "No null values found",
        "status": "PASS" if not has_nulls else "WARN",
        "failed_count": 1 if has_nulls else 0,
        "total_count": result['columns_checked'],
        "pass_rate": 0.0 if has_nulls else 1.0,
        "severity": "low",
        "checked_at": current_timestamp,
    })

print(f"Created {len(dq_check_records)} DQ check records from {len(quality_results)} tables")

# Save to Delta table
if dq_check_records:
    spark.createDataFrame(dq_check_records).write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.dq_check_results")
    
    print(f"✅ Saved {len(dq_check_records)} rows to {CATALOG}.{SCHEMA}.dq_check_results")
    
    # Display sample
    print("\nSample DQ Check Results:")
    display(spark.sql(f"""
        SELECT 
            table_name,
            check_name,
            status,
            ROUND(pass_rate * 100, 1) as pass_rate_pct,
            check_description,
            checked_at
        FROM {CATALOG}.{SCHEMA}.dq_check_results
        ORDER BY table_name, check_name
    """))
else:
    print("No DQ check records to save")

8.5 SAVING DATA QUALITY RESULTS
Created 15 DQ check records from 5 tables
✅ Saved 15 rows to nyc_mobility.gold.dq_check_results

Sample DQ Check Results:


table_name,check_name,status,pass_rate_pct,check_description,checked_at
dim_date,Duplicate Check,PASS,100.0,Duplicate groups in dim_date,2026-09-25T10:51:51.945Z
dim_date,Null Check,PASS,100.0,No null values found,2026-09-25T10:51:51.945Z
dim_date,Row Count,PASS,100.0,Total rows in dim_date,2026-09-25T10:51:51.945Z
dim_hour,Duplicate Check,PASS,100.0,Duplicate groups in dim_hour,2026-09-25T10:51:51.945Z
dim_hour,Null Check,PASS,100.0,No null values found,2026-09-25T10:51:51.945Z
dim_hour,Row Count,PASS,100.0,Total rows in dim_hour,2026-09-25T10:51:51.945Z
dim_weather,Duplicate Check,PASS,100.0,Duplicate groups in dim_weather,2026-09-25T10:51:51.945Z
dim_weather,Null Check,PASS,100.0,No null values found,2026-09-25T10:51:51.945Z
dim_weather,Row Count,PASS,100.0,Total rows in dim_weather,2026-09-25T10:51:51.945Z
dim_zone,Duplicate Check,PASS,100.0,Duplicate groups in dim_zone,2026-09-25T10:51:51.945Z
